# DhikrSpeech — end-to-end pipeline

The whole training pipeline in one notebook: from the raw recordings on Drive to the quantised TensorFlow Lite model the Android app ships. It is the five staged notebooks (`01`–`05`) combined, run top to bottom, sharing a **single Setup cell**.

Run **Setup** once, then work through the stages in order:

1. **Dataset** — inspect and validate the recordings.
2. **Preprocessing** — condition to 16 kHz mono, freeze the train/val/test split, write the manifest.
3. **Training** — train the DS-CNN (TensorBoard, checkpoints, resume).
4. **Evaluation** — metrics, confusion matrix, ROC, error analysis.
5. **Export** — SavedModel + TFLite variants, benchmarked and verified.

Stages hand off through files on Drive (manifest, checkpoints, reports), not in-memory state, so after Setup you can also jump to and run a single stage on its own. The individual `01`–`05` notebooks remain available if you prefer to run the stages as separate notebooks.

**Runtime → Change runtime type → GPU** before training, or it falls back to CPU.

In [ ]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
REPO_BRANCH = os.environ.get("DHIKR_BRANCH", "main")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def sync_clone(target: Path) -> None:
    """Pull the latest code into an existing clone.

    Without this a runtime that cloned the repository earlier keeps running that
    old copy for the rest of the session, so fixes never arrive.
    """
    subprocess.run(
        ["git", "-C", str(target), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True
    )
    subprocess.run(
        ["git", "-C", str(target), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned or updated as needed."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            # Only ever update the throwaway clone this notebook created. A repo
            # you checked out yourself is left alone - resetting it would discard
            # whatever branch and local edits you are working on.
            if IN_COLAB and candidate.parent == Path("/content/SaloAleh"):
                try:
                    sync_clone(candidate.parent)
                except Exception as error:
                    print("could not update the clone, using it as is:", error)
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        print("cloning", REPO_URL, "@", REPO_BRANCH)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)],
            check=True,
        )
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# A kernel that already imported src/ keeps the old modules even after the clone
# is updated, so drop them and let the imports below load the new code.
stale = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale:
    del sys.modules[name]

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

revision = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip()

print("project root :", PROJECT_ROOT)
print("code version :", revision or "(not a git checkout)")
print("config       :", CONFIG_PATH)
if stale:
    print("note         : reloaded %d cached src modules — re-run this notebook "
          "from the top so every stage uses the new code" % len(stale))
print()
print(config.summary())


# 01 · Dataset explorer

Reads the recordings on Drive and answers three questions before a single epoch is trained:

1. **How much data is there?** counts, per-class distribution, durations.
2. **Is any of it broken?** corrupted, empty, silent, stereo, wrong sample rate, duplicated.
3. **Is it balanced enough to train on?** thin classes are flagged.

Nothing is modified here — this notebook only reads. Cleaning happens in `02_preprocessing`.

Expected layout on Drive:

```
MyDrive/Dhikr Speech Dataset/
├── dataset/001/*.wav        one folder per phrase id
├── dataset/unknown/*.wav    filler / out-of-vocabulary audio
└── phrases.json             [{"id": 1, "text": "سبحان الله"}, ...]
```


## 1 · Locate the dataset

Paths come from `configs/config.yaml`. Change `paths.drive_root` / `paths.project_dir` there if
your dataset lives somewhere else — never edit paths in the notebooks.

In [ ]:
from pathlib import Path

paths = config.paths
rows = [
    ("dataset", paths.dataset_path),
    ("phrases.json", paths.phrases_path),
    ("processed", paths.processed_path),
    ("checkpoints", paths.checkpoints_path),
    ("exports", paths.exports_path),
    ("logs", paths.logs_path),
    ("reports", paths.reports_path),
    ("noise (optional)", paths.noise_path),
]
for name, path in rows:
    print("%-18s %-6s %s" % (name, "ok" if Path(path).exists() else "MISSING", path))

if not paths.dataset_path.is_dir():
    raise FileNotFoundError(
        "dataset folder not found: %s\n"
        "Upload your recordings there, or point paths.drive_root / paths.project_dir "
        "at the right place in configs/config.yaml." % paths.dataset_path
    )


## 2 · Phrases

`phrases.json` maps a class id to the Arabic phrase. Folder `001` is phrase id `1`.

In [ ]:
import pandas as pd

from src.dataset import load_phrases

phrases = load_phrases(paths.phrases_path)
phrase_table = pd.DataFrame(
    [
        {
            "id": phrase.id,
            "folder": phrase.folder,
            "text": phrase.text,
            # False here means the phrase is excluded by classes.include_phrases,
            # so no clip of it reaches the manifest or the model.
            "trained": config.classes.selects(phrase.folder, paths.unknown_class),
            "folder exists": (paths.dataset_path / phrase.folder).is_dir(),
            "recordings": len(list((paths.dataset_path / phrase.folder).glob("*")))
            if (paths.dataset_path / phrase.folder).is_dir()
            else 0,
        }
        for phrase in phrases
    ]
)
print("%d phrases declared" % len(phrases))
if config.classes.enabled:
    print("classes.include_phrases restricts training to %s"
          % (config.classes.include_phrases,))
phrase_table


## 3 · Index every recording

Folders are the class vocabulary: numeric folders are phrases, `unknown` is the filler class that
teaches the model to stay quiet on everything else.

In [ ]:
from src.dataset import scan_dataset

index = scan_dataset(
    paths.dataset_path,
    phrases,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
    classes=config.classes,
)

counts = index.counts()
count_table = pd.DataFrame(
    [
        {
            "class": label,
            "recordings": count,
            "share": count / max(len(index), 1),
            "text": index.label_text(label),
        }
        for label, count in counts.items()
    ]
).sort_values("recordings", ascending=False)

print("recordings :", len(index))
print("classes    :", index.num_classes)
count_table


## 4 · Validate

Every file is opened and decoded. This is the slow cell — a few minutes for thousands of clips —
and it is what finds silent takes and duplicated uploads.

Set `DEEP = False` to only read file headers (fast, but skips silence and duplicate detection).

In [ ]:
from src.dataset import validate_dataset

DEEP = True

def progress(done: int, total: int, every: int = 200) -> None:
    """Print progress without flooding the notebook output."""
    if done == total or done % every == 0:
        print("  %d / %d" % (done, total), flush=True)

report = validate_dataset(index, config.audio, deep=DEEP, progress=progress)
print()
print(report.summary())


### Issues found

| kind | meaning | what to do |
|---|---|---|
| `corrupted` | the file cannot be decoded | delete or re-record |
| `empty` | zero-length file | delete |
| `silent` | RMS below `audio.silence_dbfs` | delete — it teaches the model nothing |
| `stereo` | more than one channel | harmless, notebook 02 downmixes it |
| `sample_rate` | not 16 kHz | harmless, notebook 02 resamples it |
| `too_short` / `too_long` | outside `audio.min_duration` / `max_duration` | review; usually a truncated take |
| `duplicate` | identical audio to another file | delete the copy — duplicates leak across the train/test split |

`corrupted`, `empty` and `silent` files are excluded from preprocessing automatically.

In [ ]:
issues = report.issues_dataframe()
if len(issues):
    display(issues.groupby("kind").size().rename("count").to_frame())
    display(issues.head(50))
else:
    print("no issues found")

unusable = report.unusable_paths()
print()
print("%d file(s) will be excluded from preprocessing" % len(unusable))
for path in unusable[:20]:
    print("  ", path)


## 5 · Statistics and distribution

In [ ]:
from src import visualization as viz

stats = report.stats
print("total files    :", stats.total_files)
print("total classes  :", stats.total_classes)
print("total audio    : %.1f minutes" % (stats.total_duration / 60.0))
print("duration (s)   : mean %.2f | median %.2f | min %.2f | max %.2f" % (
    stats.mean_duration, stats.median_duration, stats.min_duration, stats.max_duration))

MIN_PER_CLASS = 50  # rule of thumb for a usable keyword-spotting class

figure = viz.plot_class_distribution(counts, highlight_below=MIN_PER_CLASS)
viz.save_figure(figure, paths.reports_path / "01_class_distribution.png")

durations = [item.duration for item in report.file_stats if item.ok and item.duration > 0]
figure = viz.plot_duration_histogram(
    durations,
    min_duration=config.audio.min_duration,
    max_duration=config.audio.max_duration,
)
viz.save_figure(figure, paths.reports_path / "01_duration_histogram.png")

thin = [label for label, count in counts.items() if count < MIN_PER_CLASS]
if thin:
    print()
    print("classes with fewer than %d recordings:" % MIN_PER_CLASS, ", ".join(thin))


## 6 · Listen to one sample per class

A quick ear check catches problems no validator can: the wrong phrase in a folder, a clipped
microphone, background speech.

In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import load_audio
from src.features import LogMelExtractor

rng = np.random.default_rng(config.seed)
extractor = LogMelExtractor(config.features, config.audio.sample_rate)
by_class = index.by_class()

previews, titles = [], []
for label in index.class_names:
    samples_for_class = by_class.get(label, [])
    if not samples_for_class:
        continue
    chosen = samples_for_class[int(rng.integers(len(samples_for_class)))]
    try:
        clip = load_audio(chosen.path, config.audio.sample_rate)
    except Exception as error:
        print("could not load", chosen.path, error)
        continue
    print("%-10s %s" % (label, chosen.path.name))
    display(Audio(clip, rate=config.audio.sample_rate))
    previews.append(extractor(clip))
    titles.append(label)

if previews:
    figure = viz.plot_feature_grid(previews, titles, hop_ms=config.features.hop_ms)
    viz.save_figure(figure, paths.reports_path / "01_class_previews.png")


## 7 · Save the validation report

Written to `reports/` on Drive:

* `validation_report.json` — statistics, issue counts, every issue
* `validation_report_files.csv` — one row per file (duration, rate, channels, RMS, content hash)

Notebook 02 reads this file to know which recordings to skip.

In [ ]:
written = report.save(paths.reports_path)
for kind, path in written.items():
    print("%-6s %s" % (kind, path))

print()
if report.is_clean:
    print("dataset is clean — continue with 02_preprocessing.ipynb")
else:
    print("%d issue(s) recorded." % len(report.issues))
    print("Delete the duplicates and silent takes, then re-run this notebook.")
    print("Everything else is handled automatically by 02_preprocessing.ipynb.")


# 02 · Preprocessing

Turns the raw recordings into the exact tensors the model trains on, and freezes the train/val/test
split so every later notebook sees the same data.

Each recording is:

1. decoded and resampled to **16 kHz mono**,
2. **silence trimmed** (`audio.trim`),
3. **loudness normalised** to a target RMS (`audio.normalize`),
4. **fitted to `audio.clip_seconds`** by padding or cropping,
5. written as **PCM16 WAV** under `processed/<class>/`.

The result is `processed/manifest.csv` — the single input of notebooks 03, 04 and 05.

Re-running is cheap: existing files are skipped unless `OVERWRITE = True`.

## 1 · Index the dataset and load the validation report

If `01_dataset.ipynb` has been run, its report tells us which files to exclude. Otherwise a quick
header-only validation is run here.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.dataset import load_phrases, scan_dataset, validate_dataset

paths = config.paths
phrases = load_phrases(paths.phrases_path)
index = scan_dataset(
    paths.dataset_path,
    phrases,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
    classes=config.classes,
)
print("indexed %d recordings across %d classes" % (len(index), index.num_classes))

report_path = paths.reports_path / "validation_report.json"
BLOCKING = {"corrupted", "empty", "silent"}

if report_path.is_file():
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    unusable = sorted({item["path"] for item in payload["issues"] if item["kind"] in BLOCKING})
    print("loaded validation report from %s" % report_path)
else:
    print("no validation report found — running a quick header-only validation")
    quick = validate_dataset(index, config.audio, deep=False)
    unusable = quick.unusable_paths()

print("%d recording(s) excluded as unusable" % len(unusable))


## 2 · Write conditioned copies

Output goes to `processed/` on Drive. Set `OVERWRITE = True` after changing any `audio.*` setting —
otherwise clips written with the old settings are kept.

In [ ]:
from src.dataset import preprocess_dataset

OVERWRITE = False

def progress(done: int, total: int, every: int = 200) -> None:
    """Print progress without flooding the notebook output."""
    if done == total or done % every == 0:
        print("  %d / %d" % (done, total), flush=True)

records, summary = preprocess_dataset(
    index,
    config,
    exclude=unusable,
    overwrite=OVERWRITE,
    progress=progress,
)
print()
print(summary.summary())
print("clips in manifest:", len(records))

if summary.failed:
    print()
    print("failed files are listed above and are absent from the manifest")


## 3 · Split into train / val / test

Stratified per class, so every class keeps its proportion in each split. Ratios come from
`split.*` in the config, and the seed makes the split reproducible.

Set `split.group_regex` in the config when several recordings share a speaker — whole groups then
move together, which stops a speaker appearing in both train and test.

In [ ]:
from src.dataset import assign_splits, save_manifest, split_counts

records = assign_splits(records, config.split, config.seed)
counts = split_counts(records)
print("split sizes:", counts)

frame = pd.DataFrame([
    {"class": record.label, "split": record.split} for record in records
])
pivot = frame.pivot_table(index="class", columns="split", aggfunc=len, fill_value=0)
display(pivot)

manifest_path = save_manifest(records, paths.manifest_path)
print()
print("manifest written to", manifest_path)


## 4 · Preview the front-end

What the model actually sees: a `(frames, mel bins)` log mel spectrogram. Shape and framing come
from `features.*`, and this exact geometry is what the exported TFLite model expects.

In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import load_audio, read_wav
from src.features import LogMelExtractor
from src import visualization as viz

extractor = LogMelExtractor(config.features, config.audio.sample_rate)
frames, mel_bins, channels = config.input_shape
print("model input: %d frames x %d mel bins x %d channel" % (frames, mel_bins, channels))

rng = np.random.default_rng(config.seed)
previews, titles = [], []
for label in sorted({record.label for record in records}):
    for_class = [record for record in records if record.label == label]
    chosen = for_class[int(rng.integers(len(for_class)))]
    clip, _ = read_wav(chosen.resolve(paths.processed_path))
    previews.append(extractor(clip))
    titles.append("%s · %s" % (label, Path(chosen.path).name))

figure = viz.plot_feature_grid(previews[:9], titles[:9], hop_ms=config.features.hop_ms)
viz.save_figure(figure, paths.reports_path / "02_feature_previews.png")

example = records[int(rng.integers(len(records)))]
raw = load_audio(example.source_path, config.audio.sample_rate)
conditioned, _ = read_wav(example.resolve(paths.processed_path))
print()
print("before / after conditioning:", Path(example.source_path).name)
display(Audio(raw, rate=config.audio.sample_rate))
display(Audio(conditioned, rate=config.audio.sample_rate))
viz.save_figure(
    viz.plot_waveform(conditioned, config.audio.sample_rate, title="conditioned clip"),
    paths.reports_path / "02_conditioned_waveform.png",
)


## 5 · Preview the augmentation

Augmentation runs **on the fly during training only** — nothing here is written to Drive. This cell
shows what each transform does to one clip so the ranges in `augmentation.*` can be sanity checked
by eye and by ear.

If `noise/` on Drive is empty, background noise falls back to synthetic white/pink noise. Dropping
real room recordings in there is the single cheapest accuracy win for a phone-deployed model.

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor, spec_augment

noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
print("noise clips available:", len(noise_bank))

augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)
base, _ = read_wav(records[0].resolve(paths.processed_path))

# Force one transform at a time by disabling the others.
def only(name: str):
    single = config.with_overrides({
        "augmentation.%s.probability" % other: 0.0
        for other in ["background_noise", "pitch_shift", "speed_perturb", "gain", "time_shift"]
        if other != name
    }).with_overrides({"augmentation.%s.probability" % name: 1.0})
    return WaveformAugmentor(single.augmentation, single.audio, noise_bank)

variants = [("original", base)]
for name in ["background_noise", "pitch_shift", "speed_perturb", "gain", "time_shift"]:
    variants.append((name, only(name)(base, np.random.default_rng(config.seed))))

for name, clip in variants:
    print(name)
    display(Audio(clip, rate=config.audio.sample_rate))

panels = [extractor(clip) for _, clip in variants]
labels = [name for name, _ in variants]
panels.append(spec_augment(
    extractor(base),
    config.with_overrides({"augmentation.spec_augment.probability": 1.0}).augmentation,
    np.random.default_rng(config.seed),
))
labels.append("spec_augment")

figure = viz.plot_feature_grid(panels, labels, hop_ms=config.features.hop_ms)
viz.save_figure(figure, paths.reports_path / "02_augmentation_previews.png")


## 6 · Global feature statistics (optional)

Only needed when `features.normalize` is `global`. The default, `per_example`, normalises each clip
on its own and needs no dataset statistics — which also makes the Android port simpler.

In [ ]:
from src.features import compute_global_stats

if config.features.normalize == "global":
    train_records = [record for record in records if record.split == "train"]
    sample_records = train_records[: min(len(train_records), 2000)]
    raw_extractor = LogMelExtractor(
        config.with_overrides({"features.normalize": "none"}).features,
        config.audio.sample_rate,
    )
    stats = compute_global_stats(
        raw_extractor(read_wav(record.resolve(paths.processed_path))[0])
        for record in sample_records
    )
    stats_path = stats.save(paths.processed_path / "feature_stats.json")
    print("global stats over %d clips:" % len(sample_records), stats)
    print("written to", stats_path)
else:
    print("features.normalize = %r — no global statistics needed" % config.features.normalize)


## 7 · Done

Written to Drive:

* `processed/<class>/*.wav` — conditioned 16 kHz mono PCM16 clips
* `processed/manifest.csv` — path, class, phrase id, split for every clip
* `reports/02_*.png` — front-end and augmentation previews

Continue with `03_training.ipynb`.

In [ ]:
print("processed clips :", len(records))
print("splits          :", split_counts(records))
print("manifest        :", paths.manifest_path)
print("processed root  :", paths.processed_path)


# 03 · Training

Trains the DS-CNN phrase spotter on the manifest written by `02_preprocessing`.

Enabled by the config, not by editing this notebook: TensorBoard, mixed precision, early stopping,
checkpointing, resume, class weights, label smoothing, LR schedule, automatic train/val split,
fixed seed, batch size, epochs and optimizer.

**Resuming.** Re-run this notebook with the same `RUN_NAME`; `BackupAndRestore` picks the run up at
the epoch it stopped at, optimizer state included. Set a new `RUN_NAME` to start fresh.

**Runtime → Change runtime type → GPU** before running, or training falls back to CPU.

## 1 · Seed, precision, device

In [ ]:
import tensorflow as tf

from src.trainer import configure_mixed_precision, set_global_seed

RUN_NAME = config.model.name  # change to start a separate run

# `training.resume: true` means re-running this notebook restores the previous
# run's weights, optimiser state and epoch counter. That is what you want after a
# Colab disconnect, and exactly what you do not want after changing a
# hyperparameter - the change would be applied on top of the old model and the
# printed history would splice both runs together. Set this to True whenever the
# config changed since the last run.
FRESH_START = False

set_global_seed(config.seed)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed = configure_mixed_precision(config.training.mixed_precision)
print("TensorFlow  :", tf.__version__)
print("GPU         :", [gpu.name for gpu in gpus] or "none — training on CPU")
print("mixed float16:", mixed)
print("run name    :", RUN_NAME)


## 2 · Load the manifest

In [ ]:
import pandas as pd

from src.dataset import class_names_from_manifest, filter_split, load_manifest, split_counts

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

train_records = filter_split(records, "train")
val_records = filter_split(records, "val")
test_records = filter_split(records, "test")

if not train_records or not val_records:
    raise ValueError("training needs a non-empty train and val split — re-run 02_preprocessing")

print("classes :", len(class_names), class_names)
print("splits  :", split_counts(records))

display(pd.DataFrame(
    [{"class": record.label, "split": record.split} for record in records]
).pivot_table(index="class", columns="split", aggfunc=len, fill_value=0))

# A split of a few clips per class cannot measure a model, and a handful of clips
# per class cannot train one. Say so here rather than after an hour of training.
val_per_class = len(val_records) / len(class_names)
train_per_class = len(train_records) / len(class_names)
print()
print("train clips / class : %.1f" % train_per_class)
print("val clips / class   : %.1f  (val accuracy moves in steps of %.2f)"
      % (val_per_class, 1.0 / max(len(val_records), 1)))
if val_per_class < 5 or train_per_class < 30:
    print()
    print("!! this dataset is too small to train or to measure a %d-class model."
          % len(class_names))
    print("   Aim for 50-100+ recordings per class from 10+ speakers for a first")
    print("   usable model. Below that, expect the model to collapse to a single")
    print("   class and validation accuracy to sit at chance (%.4f)."
          % (1.0 / len(class_names)))
    print("   The sanity check in section 6b still tells you whether the pipeline")
    print("   itself is correct, which is the useful thing to know meanwhile.")

## 3 · Input pipeline

Training clips are decoded once and cached, then re-augmented every epoch. Validation is never
augmented and never shuffled.

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor
from src.dataset import make_tf_dataset
from src.features import FeatureStats, LogMelExtractor

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None

extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)
noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)

train_dataset = make_tf_dataset(train_records, config, extractor, training=True, augmentor=augmentor)
val_dataset = make_tf_dataset(val_records, config, extractor, training=False)

features_batch, labels_batch = next(iter(train_dataset))
print("features:", features_batch.shape, features_batch.dtype)
print("labels  :", labels_batch.shape, labels_batch.dtype)
print("expected:", config.input_shape)
assert tuple(features_batch.shape[1:]) == config.input_shape


## 4 · Class weights

Balanced weights counteract an uneven dataset, so a class with 60 recordings still matters as much
as one with 600. Disable with `training.class_weights: false`.

In [ ]:
from src.dataset import compute_class_weights

class_weight = compute_class_weights(
    [record.class_index for record in train_records], len(class_names)
)
display(pd.DataFrame(
    [
        {
            "class": class_names[index],
            "train clips": sum(1 for record in train_records if record.class_index == index),
            "weight": round(weight, 3),
        }
        for index, weight in sorted(class_weight.items())
    ]
))
print("class weights enabled:", config.training.class_weights)


## 5 · Build the model

DS-CNN: a strided convolutional stem followed by depthwise separable blocks. Small enough for a
phone, and it quantises to INT8 without a Flex delegate.

In [ ]:
from src.models import build_model, estimate_flops, model_summary_text

model = build_model(config.input_shape, len(class_names), config.model)
model.summary()

flops = estimate_flops(model)
print()
print("parameters     : %s" % format(model.count_params(), ","))
print("approx MFLOPs  : %.1f per inference" % (flops / 1e6))
(paths.checkpoints_path / RUN_NAME).mkdir(parents=True, exist_ok=True)
(paths.checkpoints_path / RUN_NAME / "model_summary.txt").write_text(
    model_summary_text(model), encoding="utf-8"
)


## 6 · TensorBoard

Run this before `fit` and the charts update live during training.

In [ ]:
LOG_DIR = paths.logs_path / RUN_NAME / "tensorboard"
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("log dir:", LOG_DIR)

try:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", "--logdir '%s'" % LOG_DIR)
except Exception as error:  # not running under IPython
    print("start TensorBoard manually:  tensorboard --logdir '%s'" % LOG_DIR)
    print(error)


## 6b · Sanity check — can this pipeline learn at all?

Before spending an hour on a real run, prove that the model can **memorise a handful of clips**.
A few dozen unaugmented recordings, a fresh copy of the model, a couple of hundred steps: training
accuracy has to go to ~1.0. It only has to overfit, so anything less means the fault is upstream of
the hyperparameters — the features, the labels, or the model itself.

This is the test to run first whenever a run sits at chance (0.10 for 10 classes) and the model
predicts one class for everything.


In [ ]:
from src.dataset import make_tf_dataset
from src.trainer import sanity_overfit, sanity_overfit_report

SANITY_CLIPS = 40          # clips to memorise, a few per class
SANITY_STEPS = 200         # optimiser steps to do it in
RUN_SANITY_CHECK = True

if RUN_SANITY_CHECK:
    # Take a stratified handful: at least one clip per class, otherwise the test
    # can pass on a subset that happens to be a single class.
    by_class = {}
    for record in train_records:
        by_class.setdefault(record.class_index, []).append(record)
    subset = []
    position = 0
    while len(subset) < min(SANITY_CLIPS, len(train_records)):
        added = False
        for index in sorted(by_class):
            if position < len(by_class[index]) and len(subset) < SANITY_CLIPS:
                subset.append(by_class[index][position])
                added = True
        if not added:
            break
        position += 1

    # training=False -> no augmentation. Memorising augmented clips is a different,
    # much harder test and not what we are asking here.
    sanity_dataset = make_tf_dataset(
        subset, config, extractor, training=False, batch_size=min(16, len(subset)), shuffle=False
    )
    sanity = sanity_overfit(model, sanity_dataset, steps=SANITY_STEPS)
    print("clips            :", len(subset))
    print(sanity_overfit_report(sanity, len(class_names)))
else:
    print("sanity check skipped")


## 7 · Train

Checkpoints, logs and the config snapshot are written to Drive as training runs, so an interrupted
Colab session loses nothing. Interrupting this cell and re-running it resumes the run.

In [ ]:
import math

from src.trainer import Trainer

steps_per_epoch = math.ceil(len(train_records) / config.training.batch_size)

trainer = Trainer(
    config=config,
    model=model,
    num_classes=len(class_names),
    steps_per_epoch=steps_per_epoch,
    run_name=RUN_NAME,
)

if FRESH_START:
    trainer.reset_run()

trainer.compile()

total_steps = steps_per_epoch * config.training.epochs
print("steps per epoch :", steps_per_epoch)
print("total steps     :", total_steps)
if total_steps < 2000:
    # Convergence follows gradient steps, not epochs. Under a couple of thousand,
    # a DS-CNN trained from scratch is still near its initialisation: the loss
    # falls a little each epoch and accuracy looks pinned near chance.
    print("                  !! under 2000 steps - expect an undertrained model.")
    print("                     Lower training.batch_size or raise training.epochs.")
print("epochs          :", config.training.epochs)
print("checkpoints     :", trainer.checkpoint_dir)
print("resume enabled  :", config.training.resume)
print("resuming a run  :", trainer.is_resuming)
print()

artifacts = trainer.fit(train_dataset, val_dataset, class_weight=class_weight)
print()
print(artifacts.summary())


## 8 · Training curves

In [ ]:
from src import visualization as viz

figure = viz.plot_training_history(artifacts.history, title="run: %s" % RUN_NAME)
viz.save_figure(figure, paths.reports_path / ("03_history_%s.png" % RUN_NAME))

display(pd.DataFrame(artifacts.history).tail(10))


## 9 · Quick check on the validation split

A full evaluation with per-class metrics, confusion matrix and error analysis is
`04_evaluation.ipynb`. This is only a smoke check that the best checkpoint reloads and performs.

In [ ]:
from src.metrics import evaluate_model
from src.trainer import load_trained_model

best_model = load_trained_model(artifacts.best_model_path)
result = evaluate_model(
    best_model,
    val_dataset,
    class_names,
    paths=[record.path for record in val_records],
    confidence_threshold=config.evaluation.confidence_threshold,
)
print(result.summary())
print()
print("predicted class distribution (a healthy model spreads across all classes):")
for label, count in result.prediction_distribution().items():
    print("  %-10s %d" % (label, count))
print()
print("best checkpoint:", artifacts.best_model_path)
print("continue with 04_evaluation.ipynb")


# 04 · Evaluation

Scores the best checkpoint on a split it never trained on and writes every chart and table to
`reports/` on Drive.

Produced here: accuracy, precision, recall, F1 (macro and weighted), a confusion matrix, per-class
metrics, one-vs-rest ROC with macro AUC, a false-positive and false-negative breakdown, and a
confidence-threshold sweep for the on-device reject gate.

Which split is used comes from `evaluation.split` (default `test`).

## 1 · Load the model and the evaluation split

In [ ]:
import pandas as pd

from src.dataset import class_names_from_manifest, filter_split, load_manifest, make_tf_dataset
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

split_name = config.evaluation.split
eval_records = filter_split(records, split_name)
if not eval_records:
    fallback = "val"
    print("split %r is empty — falling back to %r" % (split_name, fallback))
    split_name, eval_records = fallback, filter_split(records, fallback)
if not eval_records:
    raise ValueError("no clips to evaluate — re-run 02_preprocessing")

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

# No shuffling and no augmentation: predictions line up with eval_records.
eval_dataset = make_tf_dataset(
    eval_records,
    config,
    extractor,
    training=False,
    batch_size=config.evaluation.batch_size,
    shuffle=False,
)

print("checkpoint :", checkpoint)
print("split      : %s (%d clips)" % (split_name, len(eval_records)))
print("classes    :", len(class_names))


## 2 · Predict and score

In [ ]:
from src.metrics import evaluate_model

result = evaluate_model(
    model,
    eval_dataset,
    class_names,
    paths=[record.path for record in eval_records],
    confidence_threshold=config.evaluation.confidence_threshold,
)
print(result.summary())

macro, weighted = result.averaged("macro"), result.averaged("weighted")
display(pd.DataFrame([
    {"average": "macro", **{key: round(value, 4) for key, value in macro.items()}},
    {"average": "weighted", **{key: round(value, 4) for key, value in weighted.items()}},
]))


## 3 · Per-class metrics

Sorted worst first — the classes at the top are the ones that need more recordings.

In [ ]:
from src import visualization as viz

per_class = result.per_class()
table = result.to_dataframe().sort_values("f1")
display(table)

figure = viz.plot_per_class_metrics(per_class)
viz.save_figure(figure, paths.reports_path / "04_per_class_metrics.png")

weak = table[table["f1"] < 0.8]
if len(weak):
    print("classes below 0.80 F1:", ", ".join(weak["label"].tolist()))
else:
    print("every class is at or above 0.80 F1")


## 4 · Confusion matrix

Rows are the true class, columns the prediction. Bright cells off the diagonal are the phrase pairs
the model mixes up — usually phrases that share a leading word.

In [ ]:
matrix = result.confusion_matrix

figure = viz.plot_confusion_matrix(matrix, class_names, normalize=True,
                                   title="confusion matrix (row-normalised)")
viz.save_figure(figure, paths.reports_path / "04_confusion_matrix.png")

figure = viz.plot_confusion_matrix(matrix, class_names, normalize=False,
                                   title="confusion matrix (counts)")
viz.save_figure(figure, paths.reports_path / "04_confusion_matrix_counts.png")

confusions = result.top_confusions(config.evaluation.top_k_confusions)
if confusions:
    display(pd.DataFrame(confusions, columns=["true", "predicted", "count"]))
else:
    print("no off-diagonal errors")


## 5 · ROC

One-vs-rest per class. A class with no examples in this split is skipped, since its AUC is
undefined.

In [ ]:
if config.evaluation.roc:
    curves = result.roc_curves()
    macro_auc = curves.get("__macro__", {}).get("auc")
    print("macro AUC:", round(macro_auc, 4) if macro_auc is not None else "n/a")

    figure = viz.plot_roc_curves(curves)
    viz.save_figure(figure, paths.reports_path / "04_roc_curves.png")

    display(pd.DataFrame(
        [
            {"class": label, "auc": round(float(payload["auc"]), 4)}
            for label, payload in curves.items()
            if label != "__macro__"
        ]
    ).sort_values("auc"))
else:
    print("evaluation.roc is false — skipped")


## 6 · False positives and false negatives

* **False positive** — another phrase was predicted as this class. On device this is a phantom count.
* **False negative** — this class was said but predicted as something else. On device this is a missed count.

Listed for the classes with the most errors, with the file path so the clip can be listened to.

In [ ]:
from dataclasses import asdict

worst = sorted(per_class, key=lambda item: item.false_positives + item.false_negatives, reverse=True)
limit = config.evaluation.error_examples

for metrics in worst[:5]:
    if metrics.false_positives == 0 and metrics.false_negatives == 0:
        continue
    print("=" * 78)
    print("%s — %d false positive(s), %d false negative(s), support %d"
          % (metrics.label, metrics.false_positives, metrics.false_negatives, metrics.support))

    false_positives = result.false_positives(metrics.label, limit)
    if false_positives:
        print("\nfalse positives (predicted %s, actually something else):" % metrics.label)
        display(pd.DataFrame([asdict(case) for case in false_positives]))

    false_negatives = result.false_negatives(metrics.label, limit)
    if false_negatives:
        print("\nfalse negatives (%s said, predicted otherwise):" % metrics.label)
        display(pd.DataFrame([asdict(case) for case in false_negatives]))

if not any(item.false_positives or item.false_negatives for item in per_class):
    print("no errors on this split")


## 7 · Listen to the errors

The fastest way to tell a model problem from a data problem: if the clip sounds like the predicted
phrase, the label is wrong, not the model.

In [ ]:
from IPython.display import Audio, display

from src.audio import read_wav

errors = result.all_errors(limit=8)
if not errors:
    print("no misclassified clips to play")
for case in errors:
    print("true %-10s predicted %-10s confidence %.3f  %s"
          % (case.true_label, case.predicted_label, case.confidence, case.path))
    try:
        clip, _ = read_wav(paths.processed_path / case.path)
        display(Audio(clip, rate=config.audio.sample_rate))
    except Exception as error:
        print("  could not load:", error)


## 8 · Confidence threshold

On device the model runs continuously, so a prediction below a threshold should be discarded rather
than counted. This sweep picks that threshold: raise it until the error rate is acceptable, then
check how many correct detections it costs.

In [ ]:
import numpy as np

correct = result.y_true == result.y_pred
figure = viz.plot_confidence_distribution(
    result.confidence, correct, threshold=config.evaluation.confidence_threshold
)
viz.save_figure(figure, paths.reports_path / "04_confidence_distribution.png")

sweep = pd.DataFrame([
    result.rejection_stats(threshold) for threshold in np.arange(0.0, 1.0, 0.05)
])
sweep["accuracy_on_accepted"] = sweep["accuracy_on_accepted"].round(4)
sweep["accept_rate"] = sweep["accept_rate"].round(4)
display(sweep)

print()
print("current threshold (evaluation.confidence_threshold = %.2f):"
      % config.evaluation.confidence_threshold)
print(result.rejection_stats())


## 9 · Save the report

Written to `reports/`:

* `evaluation.json` — all metrics
* `evaluation_per_class.csv`
* `evaluation_errors.csv` — every misclassified clip
* `evaluation_confusion_matrix.csv`
* `04_*.png` — every chart above

In [ ]:
written = result.save(paths.reports_path)
for kind, path in written.items():
    print("%-18s %s" % (kind, path))

print()
print("charts:")
for path in sorted(paths.reports_path.glob("04_*.png")):
    print("  ", path)

print()
print("accuracy %.4f on the %s split — continue with 05_export.ipynb"
      % (result.accuracy, split_name))


# 05 · Export

Converts the trained checkpoint into the artefacts the Android app ships, then benchmarks and
verifies each one.

| variant | weights | activations | typical use |
|---|---|---|---|
| `float32` | float32 | float32 | reference — matches Keras exactly |
| `dynamic_range` | int8 | float32 | ~4× smaller, no calibration data needed |
| `int8` | int8 | int8 | smallest and fastest on phones; needs calibration clips |

Every variant is benchmarked (size, latency, arena estimate) and verified against the Keras model on
real clips, so a quantisation that damages accuracy is visible before it ships.

A checkpoint trained with mixed precision computes in float16, and TFLite has no float16 kernels for
`Conv2D` / `DepthwiseConv2dNative` / `Relu` — converting it directly fails with *"op is neither a
custom op nor a flex op"*. `export_all` handles this: it rebuilds the model in float32 first, which
is lossless because mixed precision keeps the master weights in float32 all along.


## 1 · Load the model and rebuild the front-end

In [ ]:
import numpy as np
import pandas as pd

from src.dataset import (
    class_names_from_manifest, filter_split, load_manifest, load_phrases, make_tf_dataset,
)
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

print("checkpoint :", checkpoint)
print("classes    :", len(class_names))
print("input shape:", config.input_shape)


## 2 · Calibration and verification clips

INT8 quantisation measures activation ranges from real features, so calibration comes from the
**training** split. Verification compares TFLite against Keras on **held-out** clips.

In [ ]:
from src.export import collect_features

train_records = filter_split(records, "train")
holdout_records = filter_split(records, "test") or filter_split(records, "val")

calibration_dataset = make_tf_dataset(
    train_records, config, extractor, training=False, batch_size=32, shuffle=False
)
calibration_features = collect_features(
    calibration_dataset, config.export.representative_samples
)

verification_features = None
if holdout_records:
    verification_dataset = make_tf_dataset(
        holdout_records, config, extractor, training=False, batch_size=32, shuffle=False
    )
    verification_features = collect_features(verification_dataset, 200)

print("calibration clips :", calibration_features.shape)
print("verification clips:", None if verification_features is None else verification_features.shape)


## 3 · Export, benchmark, verify

Conversion failures are isolated per variant: if INT8 fails, the other two are still produced.

In [ ]:
from src.export import export_all

metrics_payload = {}
evaluation_json = paths.reports_path / "evaluation.json"
if evaluation_json.is_file():
    import json

    payload = json.loads(evaluation_json.read_text(encoding="utf-8"))
    metrics_payload = {
        "accuracy": payload.get("accuracy"),
        "macro": payload.get("macro"),
        "num_samples": payload.get("num_samples"),
    }

phrases = {phrase.id: phrase.text for phrase in load_phrases(paths.phrases_path)}

bundle = export_all(
    model=model,
    config=config,
    class_names=class_names,
    frontend=extractor.metadata(),
    calibration_features=calibration_features,
    verification_features=verification_features,
    phrases=phrases,
    metrics=metrics_payload,
)

print()
print(bundle.table())


## 4 · Compare the variants

`expected_android_ms` is the measured latency multiplied by `export.android_latency_factor` — an
estimate for comparing variants, not a measurement. Measure on a real device before quoting it.

In [ ]:
from dataclasses import asdict

from src import visualization as viz

benchmarks = [item.benchmark for item in bundle.models if item.benchmark]
frame = pd.DataFrame([asdict(item) for item in benchmarks])
display(frame[[
    "name", "size_kb", "mean_latency_ms", "median_latency_ms", "p95_latency_ms",
    "arena_estimate_kb", "expected_android_ms", "input_dtype", "output_dtype",
]].round(3))

if benchmarks:
    figure = viz.plot_benchmark(benchmarks)
    viz.save_figure(figure, paths.reports_path / "05_benchmark.png")

verifications = [item.verification for item in bundle.models if item.verification]
if verifications:
    display(pd.DataFrame([asdict(item) for item in verifications]).round(5))
    for item in verifications:
        if not item.passed:
            print("WARNING: %s disagrees with the Keras model — do not ship it "
                  "without checking accuracy in 04_evaluation" % item.name)


## 5 · Front-end parameters for Android

The model takes log mel features, not raw audio, so the Android side must produce **identical**
features. These files pin that contract:

* `model_meta.json` — sample rate, clip length, FFT/window/hop, mel range, log offset, normalisation
* `mel_filterbank.json` — the exact mel matrix, so no filterbank has to be re-derived on device
* `labels.txt` — class order, one label per line
* `labels_phrases.json` — class index → phrase id → Arabic text

`README.md` has the matching Kotlin front-end.

In [ ]:
filterbank_path = extractor.save_filterbank(paths.exports_path / "mel_filterbank.json")
bundle.filterbank_path = filterbank_path

print("labels     :", bundle.labels_path)
print("metadata   :", bundle.metadata_path)
print("filterbank :", filterbank_path)
print()
for key, value in extractor.metadata().items():
    print("%-14s %s" % (key, value))


## 5b · Archive this export to history

`export_all` overwrites the export root every run, so the root always holds the **latest** model — the
one the app ships. This copies the whole export (the `.tflite` variants and their sidecars —
`labels.txt`, `model_meta.json`, `mel_filterbank.json`) into a dated snapshot under
`exports/history/<datetime>_<phrases>_<accuracy>/`, so every published model is kept and can be told
apart later. The bulky `saved_model/` is left out of snapshots.

The accuracy in the folder name is the one loaded above from `reports/evaluation.json` — re-run
**04 · Evaluation** after re-training so a snapshot is not stamped with a stale number (`accNA` means
no evaluation was found). The folder name is only a label; `model_meta.json` travels inside the
snapshot with the full metrics.

A Space pointed at the export root loads only the latest model — the fetcher skips `history/`. To
publish an older model, point the Space straight at its `history/<name>/` subfolder.


In [ ]:
from src.export import archive_export

# metrics_payload / evaluation_json come from the export cell above.
accuracy = metrics_payload.get("accuracy")
try:
    archive_dir = archive_export(
        paths.exports_path,
        include_phrases=config.classes.include_phrases,
        include_unknown=config.classes.include_unknown,
        accuracy=accuracy,
    )
except Exception as error:
    # The export root is already complete — the app ships from there. A failed
    # archive (e.g. a Google Drive FUSE copy hiccup) must not fail a good run.
    archive_dir = None
    print("could not archive this export to history:", error)

if archive_dir is not None:
    print("archived this export to:")
    print("  ", archive_dir)
    if accuracy is None:
        print("   !! accuracy unknown (accNA) — run 04 · Evaluation first so the "
              "snapshot folder is labelled with a real number")
    else:
        print("   accuracy %.4f from %s" % (accuracy, evaluation_json))
    print()
    for path in sorted(archive_dir.iterdir()):
        if path.is_file():
            print("    %-34s %8.1f KB" % (path.name, path.stat().st_size / 1024.0))


## 6 · What to ship

The recommendation is the smallest variant that still agrees with the Keras model. Override it if a
device measurement says otherwise.

In [ ]:
recommended = bundle.recommended()
if recommended is None:
    print("no variant passed verification — re-check the calibration clips and re-run")
else:
    print("recommended :", recommended.name)
    print("file        :", recommended.path)
    print("size        : %.2f MB" % (recommended.benchmark.size_kb / 1024.0))
    print("latency     : %.2f ms mean on this machine" % recommended.benchmark.mean_latency_ms)
    if recommended.verification:
        print("agreement   : %.2f%% with Keras" % (recommended.verification.agreement * 100))

from src.export import HISTORY_DIRNAME

print()
print("exports on Drive (the %s/ archive is listed by the cell above):" % HISTORY_DIRNAME)
history_dir = paths.exports_path / HISTORY_DIRNAME
for path in sorted(paths.exports_path.rglob("*")):
    if path.is_file() and history_dir not in path.parents:
        print("  %-34s %8.1f KB" % (path.name, path.stat().st_size / 1024.0))


## 7 · Android integration

Copy into `app/src/main/assets/`:

```
dhikr_int8.tflite      (or the recommended variant)
labels.txt
model_meta.json
mel_filterbank.json
```

On device, per inference:

1. record 16 kHz mono PCM16 into a ring buffer,
2. take the last `clip_seconds` of audio (`clip_samples` samples),
3. trim / normalise exactly as `model_meta.json.audio` describes,
4. compute the log mel spectrogram with the parameters in `model_meta.json.frontend`,
5. feed `(frames, n_mels, 1)` to the interpreter,
6. reject predictions below the threshold chosen in notebook 04, and debounce repeats so one spoken
   phrase counts once.

The full Kotlin front-end, the Gradle dependency and the threshold/debounce guidance are in
`README.md` under *Integrate into Android*.

In [ ]:
print("Export complete.\n")
print("copy to app/src/main/assets/:")
for name in ["labels.txt", "model_meta.json", "mel_filterbank.json"]:
    print("  ", paths.exports_path / name)
if recommended is not None:
    print("  ", recommended.path)
print()
print("Growing the dataset later: add recordings to dataset/<class>/ and re-run notebooks 01 → 05.")
print("Preprocessing skips clips it has already written, so re-runs only cost the new files.")
